In [ ]:
import os
import sys

SMOKE_TEST = bool(os.environ.get('MATCHA_SMOKE_TEST'))  # chạy thử vài chục câu để kiểm tra code

HF_REPO = 'doof-ferb/infore1_25hours'   # InfoRe1: ~25 giờ, 1 giọng nữ, thu sạch
TEST_FROM = 14800                        # câu từ chỉ số này trở đi để dành cho scripts/benchmark.py
MAX_SHARDS = None
MAX_UTTS = None
MIN_SECONDS, MAX_SECONDS = 1.0, 12.0
N_VAL = 100
N_WORKERS = os.cpu_count()

TOTAL_STEPS = 150_000
SNAPSHOT_EVERY = 10_000
SAMPLE_EVERY = 5_000
LEARNING_RATE = 1e-4
GRAD_CLIP = 5.0
EMA_DECAY = 0.9999                       # snapshot lưu trọng số trung bình trượt: giọng ổn định, ít rè hơn
N_EVAL_SNAPSHOTS = 6
RESUME_EVERY_MIN = 20
VOC_RESUME_EVERY_MIN = 30

VOC_N_UTTS = 4000
VOC_EVERY = 3
VOC_STEPS = 50_000
VOC_BATCH = 16
VOC_LR = 1e-4
VOC_DISC_WARMUP = 2_000
VOC_SAMPLE_EVERY = 10_000
VOC_TF_TIMESTEPS = 20

if SMOKE_TEST:
    MAX_SHARDS, MAX_UTTS, N_VAL, N_WORKERS = 1, 60, 6, 0
    TOTAL_STEPS, SNAPSHOT_EVERY, SAMPLE_EVERY, N_EVAL_SNAPSHOTS = 12, 6, 6, 2
    VOC_N_UTTS, VOC_EVERY, VOC_STEPS, VOC_BATCH = 6, 2, 6, 2
    VOC_DISC_WARMUP, VOC_SAMPLE_EVERY = 3, 6
    RESUME_EVERY_MIN = VOC_RESUME_EVERY_MIN = 0
print(f'{TOTAL_STEPS} step | smoke test: {SMOKE_TEST}')

In [ ]:
import glob
import shutil
import subprocess
import time

import torch

IS_KAGGLE = os.path.exists('/kaggle/working')
if not IS_KAGGLE and not SMOKE_TEST:
    raise RuntimeError('Notebook chạy trên Kaggle (Settings -> Accelerator: GPU T4 x2 hoặc P100, Internet: ON).')
HAS_GPU = torch.cuda.is_available()
if not HAS_GPU and not SMOKE_TEST:
    raise RuntimeError('Chưa bật GPU (Settings -> Accelerator -> GPU T4 x2 hoặc P100).')
DEVICE = 'cuda' if HAS_GPU else 'cpu'
GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3 if HAS_GPU else 0.0
if HAS_GPU:
    print(f'[GPU] {torch.cuda.get_device_name(0)} | VRAM {GPU_VRAM_GB:.1f} GB')
BATCH_SIZE = 4 if SMOKE_TEST else (64 if GPU_VRAM_GB > 30 else 32)

# Smoke test ngoài Kaggle: ghi vào thư mục chỉ định qua biến môi trường
BASE_DIR = '/kaggle/working' if IS_KAGGLE else os.environ.get('MATCHA_OUT_DIR', os.path.abspath('matcha_out'))
WORK_DIR = '/kaggle/temp' if IS_KAGGLE else f'{BASE_DIR}/work'
DATA_DIR, TRAIN_DIR, SNAPSHOT_DIR, SAMPLES_DIR, EXPORT_DIR = (
    f'{BASE_DIR}/{name}' for name in ('data', 'training', 'snapshots', 'samples', 'export')
)
for d in (DATA_DIR, TRAIN_DIR, SNAPSHOT_DIR, SAMPLES_DIR, EXPORT_DIR, WORK_DIR):
    os.makedirs(d, exist_ok=True)

DATA_PACK = f'{DATA_DIR}/infore1_train_mels.pt'
VOC_PACK = f'{DATA_DIR}/infore1_voc_wavs.pt'
META_PATH = f'{DATA_DIR}/infore1_meta.json'
VOC_TF_PACK = f'{DATA_DIR}/voc_teacher_forced.pt'
RESUME_CKPT = f'{TRAIN_DIR}/matcha_resume.pt'
VOC_RESUME = f'{TRAIN_DIR}/hifigan_resume.pt'
FINAL_CKPT = f'{EXPORT_DIR}/matcha_vi.pt'
FINAL_VOC = f'{EXPORT_DIR}/matcha_vi_hifigan.pt'
LOG_CSV = f'{BASE_DIR}/training_log.csv'
VOC_LOG_CSV = f'{BASE_DIR}/vocoder_log.csv'
CER_CSV = f'{BASE_DIR}/cer_eval_results.csv'

# Output của phiên trước được Add Data vào notebook -> chép về /kaggle/working để chạy tiếp
RESTORE = {
    DATA_DIR: ['infore1_train_mels.pt', 'infore1_voc_wavs.pt', 'infore1_meta.json', 'voc_teacher_forced.pt'],
    TRAIN_DIR: ['matcha_resume.pt', 'hifigan_resume.pt'],
    SNAPSHOT_DIR: ['step_*.pt'],
    EXPORT_DIR: ['matcha_vi.pt', 'matcha_vi_hifigan.pt'],
    BASE_DIR: ['training_log.csv', 'vocoder_log.csv', 'cer_eval_results.csv'],
}
if IS_KAGGLE:
    for dest, patterns in RESTORE.items():
        for pattern in patterns:
            for src in glob.glob(f'/kaggle/input/**/{pattern}', recursive=True):
                target = f'{dest}/{os.path.basename(src)}'
                if not os.path.exists(target):
                    shutil.copy(src, target)
                    print(f'[RESTORE] {src} -> {target}')
# Chỉ có mô hình đã xuất (không có trạng thái train) -> train tiếp từ mô hình đó
if os.path.exists(FINAL_CKPT) and not os.path.exists(RESUME_CKPT):
    shutil.copy(FINAL_CKPT, RESUME_CKPT)

print(f'{BASE_DIR} ({shutil.disk_usage(BASE_DIR).free / 1024**3:.1f} GB trống) | {DEVICE} | batch {BATCH_SIZE}')

In [ ]:
MATCHA_DIR = os.environ.get('MATCHA_DIR', f'{WORK_DIR}/Matcha-TTS')
if not os.path.exists(MATCHA_DIR):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/shivammehta25/Matcha-TTS.git', MATCHA_DIR], check=True)
    subprocess.run(['git', '-C', MATCHA_DIR, 'checkout', '-q', 'bd4d90d'], check=True)
if not SMOKE_TEST:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightning', 'hydra-core', 'conformer==0.3.2',
                    'diffusers', 'einops', 'wget', 'gdown', 'numba', 'huggingface_hub', 'pyarrow',
                    'transformers'], check=True)

# Monotonic Alignment Search bằng numba, không cần biên dịch Cython
MAS_CODE = '''import numba
import numpy as np
import torch


@numba.njit(parallel=True, cache=True)
def _maximum_path_batch(paths, values, t_xs, t_ys):
    max_neg_val = -1e9
    for b in numba.prange(paths.shape[0]):
        path, value, t_x, t_y = paths[b], values[b], t_xs[b], t_ys[b]
        for y in range(t_y):
            for x in range(max(0, t_x + y - t_y), min(t_x, y + 1)):
                v_cur = max_neg_val if x == y else value[x, y - 1]
                if x == 0:
                    v_prev = 0.0 if y == 0 else max_neg_val
                else:
                    v_prev = value[x - 1, y - 1]
                value[x, y] += max(v_prev, v_cur)
        index = t_x - 1
        for y in range(t_y - 1, -1, -1):
            path[index, y] = 1
            if index != 0 and (index == y or value[index, y - 1] < value[index - 1, y - 1]):
                index -= 1


def maximum_path(value, mask):
    value = value * mask
    device, dtype = value.device, value.dtype
    value_np = value.detach().float().cpu().numpy()
    mask_np = mask.detach().cpu().numpy()
    path = np.zeros_like(value_np, dtype=np.int32)
    t_x = mask_np.sum(1)[:, 0].astype(np.int32)
    t_y = mask_np.sum(2)[:, 0].astype(np.int32)
    _maximum_path_batch(path, value_np, t_x, t_y)
    return torch.from_numpy(path).to(device=device, dtype=dtype)
'''
with open(f'{MATCHA_DIR}/matcha/utils/monotonic_align/__init__.py', 'w', encoding='utf-8') as f:
    f.write(MAS_CODE)

for p in (MATCHA_DIR, os.getcwd()):
    if p not in sys.path:
        sys.path.insert(0, p)
from matcha.models.matcha_tts import MatchaTTS

print('[OK] Matcha-TTS')

In [ ]:
%%writefile vi_tokenizer.py
"""Bộ tách token văn bản tiếng Việt cho Matcha-TTS, dùng chung cho notebook huấn luyện và engine suy luận.

Mỗi âm tiết được tách thành các chữ cái (đã bỏ dấu thanh) và một token thanh điệu ở cuối:
"hồ" -> h, ô, 2. Sáu thanh dùng chung cho mọi nguyên âm; dấu câu (, . ? !) được giữ làm token
khi dữ liệu huấn luyện có dấu câu.

Notebook notebooks/train_matcha.ipynb chứa bản sao nguyên văn file này (kiểm tra bởi
tests/test_vi_tokenizer.py). Sau khi sửa file này, chạy `python scripts/sync_notebook_tokenizer.py`.
"""

import re
import unicodedata
from typing import Dict, List

# Tên chữ cái tiếng Việt, dùng để đánh vần nhãn đáp án (A/B/C/D) và chữ viết tắt
LETTER_NAMES = {
    "a": "a", "b": "bê", "c": "xê", "d": "dê", "đ": "đê", "e": "e", "f": "ép", "g": "giê",
    "h": "hát", "i": "i", "j": "gi", "k": "ca", "l": "e lờ", "m": "em mờ", "n": "en nờ",
    "o": "o", "p": "pê", "q": "quy", "r": "e rờ", "s": "ét", "t": "tê", "u": "u", "v": "vê",
    "w": "vê kép", "x": "ích", "y": "i dài", "z": "dét",
}
# Chữ cái ngoài bảng chữ tiếng Việt -> cách viết gần âm nhất
LOANWORD_LETTERS = (("f", "ph"), ("j", "gi"), ("w", "u"), ("z", "d"))

# Dấu thanh (dạng tổ hợp Unicode) -> token thanh điệu; "0" là thanh ngang
TONE_MARKS = {"́": "1", "̀": "2", "̉": "3", "̃": "4", "̣": "5"}
TONES = ["0", "1", "2", "3", "4", "5"]
BASE_LETTERS = list("abcdeghiklmnopqrstuvxy") + ["ă", "â", "đ", "ê", "ô", "ơ", "ư"]
PUNCTUATION = [",", ".", "?", "!"]
# Bộ token cố định; id 0 = blank/pad (chèn giữa các token như add_blank của Matcha)
SYMBOLS = ["_", " "] + PUNCTUATION + BASE_LETTERS + TONES

_PUNCT_MAP = {";": ",", ":": ",", "…": ".", "\n": "."}
_ITEM_RE = re.compile(r"[a-zăâđêôơừ-ͯ]+|[,.?!]")


def spell_out(text: str) -> str:
    """Chuẩn hóa chính tả trước khi token hóa, giữ nguyên dấu câu.

    Các bước: chuẩn Unicode NFC; thay gạch nối bằng khoảng trắng; đánh vần chữ viết tắt in hoa tối đa
    5 ký tự theo tên chữ cái ("B" -> "bê", "TTS" -> "tê tê ét"); chuyển chữ thường; thay chữ cái ngoại lai
    f/j/w/z bằng âm tiếng Việt gần nhất.

    Args:
        text: Văn bản đã qua normalizer (số, ngày tháng đã viết thành chữ).

    Returns:
        Văn bản chữ thường chỉ gồm chữ cái tiếng Việt, khoảng trắng và dấu câu.
    """
    text = unicodedata.normalize("NFC", text).replace("-", " ")
    text = re.sub(
        r"\b[A-ZĐ]{1,5}\b",
        lambda m: " ".join(LETTER_NAMES[c] for c in m.group(0).lower()),
        text,
    )
    text = text.lower()
    for src, dst in LOANWORD_LETTERS:
        text = text.replace(src, dst)
    return text


def clean_chars(text: str) -> str:
    """Đưa văn bản gốc và kết quả ASR về cùng dạng khi tính CER: chữ thường, không dấu câu, không chữ số.

    Args:
        text: Văn bản bất kỳ.

    Returns:
        Chuỗi chữ cái và một khoảng trắng giữa các từ.
    """
    text = re.sub(r"[\W\d_]+", " ", spell_out(text))
    return re.sub(r"\s+", " ", text).strip()


def syllable_tokens(syllable: str) -> List[str]:
    """Tách một âm tiết thành các chữ cái (đã bỏ dấu thanh) và một token thanh điệu ở cuối.

    Dấu phụ tạo nguyên âm (ă, â, ê, ô, ơ, ư) được giữ trong chữ cái; chỉ năm dấu thanh bị tách ra.
    Ví dụ: "hồ" -> ["h", "ô", "2"]; "gươm" -> ["g", "ư", "ơ", "m", "0"].

    Args:
        syllable: Một âm tiết chữ thường (NFC hoặc NFD).

    Returns:
        Danh sách token; phần tử cuối là thanh điệu "0" (ngang) đến "5" (nặng).
    """
    decomposed = unicodedata.normalize("NFD", syllable)
    tone = next((TONE_MARKS[c] for c in decomposed if c in TONE_MARKS), "0")
    letters = unicodedata.normalize("NFC", "".join(c for c in decomposed if c not in TONE_MARKS))
    return [c for c in letters if c in BASE_LETTERS] + [tone]


def text_to_tokens(text: str, keep_punctuation: bool = True) -> List[str]:
    """Chuyển văn bản thành danh sách token.

    Args:
        text: Văn bản đã qua normalizer.
        keep_punctuation: Giữ dấu câu (, . ? !) làm token; chỉ bật khi mô hình đã học dấu câu.
            Dấu ; : được quy về dấu phẩy, dấu … và xuống dòng quy về dấu chấm.

    Returns:
        Danh sách token; các âm tiết cách nhau bởi token khoảng trắng " ".
    """
    text = spell_out(text)
    for src, dst in _PUNCT_MAP.items():
        text = text.replace(src, dst)
    text = unicodedata.normalize("NFD", text)
    tokens: List[str] = []
    for item in _ITEM_RE.findall(text):
        if item in PUNCTUATION:
            if keep_punctuation and tokens and tokens[-1] not in PUNCTUATION:
                tokens += [" ", item]
            continue
        if tokens:
            tokens.append(" ")
        tokens += syllable_tokens(item)
    while tokens and tokens[0] in PUNCTUATION + [" "]:
        tokens.pop(0)
    return tokens


def tokens_to_ids(tokens: List[str], symbol2id: Dict[str, int]) -> List[int]:
    """Ánh xạ token sang id và chèn blank (id 0) xen giữa, như tùy chọn add_blank=True của Matcha-TTS.

    Token không có trong bộ từ vựng bị bỏ qua.

    Args:
        tokens: Kết quả của `text_to_tokens`.
        symbol2id: Bảng token -> id của checkpoint.

    Returns:
        Danh sách id độ dài 2·n + 1, dạng [0, t1, 0, t2, ..., 0].
    """
    ids = [symbol2id[t] for t in tokens if t in symbol2id]
    result = [0] * (2 * len(ids) + 1)
    result[1::2] = ids
    return result

In [ ]:
import copy
import importlib
import json
import math
import random

import numpy as np
import soundfile as sf
from omegaconf import OmegaConf
from matcha.hifigan.config import v1 as HIFIGAN_CONFIG
from matcha.hifigan.denoiser import Denoiser
from matcha.hifigan.env import AttrDict
from matcha.hifigan.models import Generator as HiFiGAN
from matcha.utils.audio import mel_spectrogram

import vi_tokenizer as vt

importlib.reload(vt)

USE_AMP = HAS_GPU
AMP_DTYPE = torch.bfloat16 if HAS_GPU and torch.cuda.get_device_capability()[0] >= 8 else torch.float16
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Tham số mel giống checkpoint LJSpeech để tái sử dụng decoder và HiFi-GAN
SAMPLE_RATE, N_FFT, N_MELS, HOP, WIN, FMIN, FMAX = 22050, 1024, 80, 256, 1024, 0, 8000

PRETRAINED_DIR = f'{WORK_DIR}/pretrained'
MATCHA_LJSPEECH_URL = 'https://github.com/shivammehta25/Matcha-TTS-checkpoints/releases/download/v1.0/matcha_ljspeech.ckpt'
HIFIGAN_UNIV_URL = 'https://github.com/shivammehta25/Matcha-TTS-checkpoints/releases/download/v1.0/g_02500000'

EVAL_SENTENCES = [
    'Hôm nay trời trong xanh, gió nhẹ thổi qua những hàng cây ven đường.',
    'Chiến thắng Điện Biên Phủ diễn ra ngày bảy tháng năm năm một nghìn chín trăm năm mươi tư.',
    'Hồ Gươm nằm ở trung tâm thủ đô Hà Nội, gắn liền với truyền thuyết vua Lê Lợi trả gươm báu.',
    'Đáp án đúng là câu bê, vì tác giả muốn nhấn mạnh tình yêu quê hương.',
    'Bạn có biết vì sao lá cây chuyển sang màu vàng vào mùa thu không?',
    'Những người nông dân cần cù đã gặt hái được một vụ mùa bội thu.',
]
if SMOKE_TEST:
    EVAL_SENTENCES = EVAL_SENTENCES[:1]


def download(url, filename):
    path = f'{PRETRAINED_DIR}/{filename}'
    if not os.path.exists(path):
        os.makedirs(PRETRAINED_DIR, exist_ok=True)
        torch.hub.download_url_to_file(url, path)
    return path


def compute_mel(wav, fmax=FMAX):
    return mel_spectrogram(wav if wav.dim() == 2 else wav[None], N_FFT, N_MELS, SAMPLE_RATE, HOP, WIN, FMIN, fmax)


def text_ids(text):
    return vt.tokens_to_ids(vt.text_to_tokens(text, keep_punctuation=meta['keep_punctuation']), SYMBOL2ID)


def build_model(hparams):
    return MatchaTTS(**OmegaConf.create(hparams))


def load_matcha(path):
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    model = build_model(ckpt['hparams'])
    model.load_state_dict(ckpt['state_dict'])
    return model.to(DEVICE).eval(), ckpt


def load_vocoder(path=None):
    path = path or download(HIFIGAN_UNIV_URL, 'hifigan_universal.pt')
    vocoder = HiFiGAN(AttrDict(HIFIGAN_CONFIG)).to(DEVICE)
    vocoder.load_state_dict(torch.load(path, map_location=DEVICE)['generator'])
    vocoder.eval()
    vocoder.remove_weight_norm()
    return vocoder, Denoiser(vocoder, mode='zeros')


@torch.no_grad()
def synthesize_vi(model, text, vocoder, denoiser=None, n_timesteps=20, temperature=0.667, length_scale=1.0):
    x = torch.tensor(text_ids(text), dtype=torch.long, device=DEVICE)[None]
    x_lengths = torch.tensor([x.shape[-1]], dtype=torch.long, device=DEVICE)
    was_training = model.training
    model.eval()
    mel = model.synthesise(x, x_lengths, n_timesteps=n_timesteps, temperature=temperature,
                           spks=None, length_scale=length_scale)['mel']
    model.train(was_training)
    audio = vocoder(mel).clamp(-1, 1)
    if denoiser is not None:
        audio = denoiser(audio.squeeze(), strength=0.00025)
    return audio.float().cpu().squeeze().numpy()


def save_atomic(obj, path):
    torch.save(obj, path + '.tmp')
    os.replace(path + '.tmp', path)


def show_audio(wav, path=None):
    if path:
        sf.write(path, wav, SAMPLE_RATE)
    try:
        from IPython.display import Audio, display

        display(Audio(wav, rate=SAMPLE_RATE))
    except ImportError:
        pass


print('|'.join(vt.text_to_tokens('Hồ Gươm, Hà Nội?')))

In [ ]:
import io
from multiprocessing import Pool

import librosa
import pyarrow.parquet as pq
from huggingface_hub import HfApi, hf_hub_download
from tqdm.auto import tqdm

HF_TOKEN = os.environ.get('HF_TOKEN')


def load_audio_bytes(raw):
    """bytes audio -> float32 mono 22.05 kHz, đã cắt im lặng và chuẩn hóa RMS ~ -20 dBFS; None nếu loại."""
    try:
        y, sr = sf.read(io.BytesIO(raw), dtype='float32', always_2d=True)
    except Exception:
        return None
    y = y.mean(axis=1)
    if sr != SAMPLE_RATE:
        y = librosa.resample(y, orig_sr=sr, target_sr=SAMPLE_RATE)
    y, _ = librosa.effects.trim(y, top_db=40)
    if not MIN_SECONDS * SAMPLE_RATE <= len(y) <= MAX_SECONDS * SAMPLE_RATE:
        return None
    y = y * (0.1 / max(1e-4, float(np.sqrt(np.mean(y ** 2)))))
    peak = float(np.abs(y).max())
    if peak > 0.95:
        y = y * (0.95 / peak)
    return y.astype(np.float32)


def iter_rows(path):
    for batch in pq.ParquetFile(path).iter_batches(batch_size=64, columns=['transcription', 'audio']):
        yield from batch.to_pylist()


if os.path.exists(DATA_PACK) and os.path.exists(META_PATH) and os.path.exists(VOC_PACK):
    data = torch.load(DATA_PACK, weights_only=False)
    with open(META_PATH, encoding='utf-8') as f:
        meta = json.load(f)
else:
    files = HfApi().list_repo_files(HF_REPO, repo_type='dataset', token=HF_TOKEN)
    shards = sorted(f for f in files if f.startswith('data/train-') and f.endswith('.parquet'))[:MAX_SHARDS]
    shard_paths = [hf_hub_download(HF_REPO, name, repo_type='dataset', token=HF_TOKEN,
                                   local_dir=f'{WORK_DIR}/hf_data') for name in tqdm(shards, desc='Tải shard')]

    items, voc_items = [], []
    mel_sum, mel_sq_sum, n_frames, row_idx = 0.0, 0.0, 0, 0
    pool = Pool(N_WORKERS) if N_WORKERS and N_WORKERS > 1 else None
    for path in shard_paths:
        if MAX_UTTS and len(items) >= MAX_UTTS:
            break
        rows = list(iter_rows(path))
        rows, row_idx = rows[:max(0, TEST_FROM - row_idx)], row_idx + len(rows)
        raws = [r['audio']['bytes'] for r in rows]
        audios = pool.imap(load_audio_bytes, raws, chunksize=8) if pool else map(load_audio_bytes, raws)
        for r, y in tqdm(zip(rows, audios), total=len(rows), desc=os.path.basename(path)):
            text = r['transcription']
            if y is None or len(vt.text_to_tokens(text)) < 2:
                continue
            mel = compute_mel(torch.from_numpy(y).to(DEVICE)).squeeze(0)
            mel_sum += mel.sum().item()
            mel_sq_sum += (mel ** 2).sum().item()
            n_frames += mel.shape[-1]
            items.append({'text': text, 'mel': mel.half().cpu()})
            if len(voc_items) < VOC_N_UTTS and len(items) % VOC_EVERY == 0:
                voc_items.append({'text': text, 'wav': torch.from_numpy((y * 32767).astype(np.int16))})
            if MAX_UTTS and len(items) >= MAX_UTTS:
                break
    if pool:
        pool.close()

    mel_mean = mel_sum / (n_frames * N_MELS)
    meta = {
        'dataset': 'infore1',
        'keep_punctuation': any(c in it['text'] for it in items for c in vt.PUNCTUATION),
        'vocab': vt.SYMBOLS,
        'mel_mean': mel_mean,
        'mel_std': float(np.sqrt(mel_sq_sum / (n_frames * N_MELS) - mel_mean ** 2)),
    }
    random.Random(1234).shuffle(items)
    data = {'val': items[:N_VAL], 'train': items[N_VAL:]}
    torch.save(data, DATA_PACK)
    torch.save(voc_items, VOC_PACK)
    with open(META_PATH, 'w', encoding='utf-8') as f:
        json.dump(meta, f, ensure_ascii=False, indent=1)

SYMBOL2ID = {s: i for i, s in enumerate(meta['vocab'])}
hours = sum(it['mel'].shape[-1] for it in data['train'] + data['val']) * HOP / SAMPLE_RATE / 3600
print(f"{len(data['train'])} câu train, {len(data['val'])} câu val ({hours:.1f} giờ), "
      f"{len(meta['vocab'])} token, dấu câu: {meta['keep_punctuation']}")

In [ ]:
from torch.utils.data import DataLoader, Dataset
from matcha.utils.model import fix_len_compatibility


class MelDataset(Dataset):
    def __init__(self, items):
        self.items = items
        self.lengths = [it['mel'].shape[-1] for it in items]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        it = self.items[i]
        return torch.tensor(text_ids(it['text']), dtype=torch.long), (it['mel'].float() - meta['mel_mean']) / meta['mel_std']


def collate(batch):
    x_lengths = torch.tensor([b[0].shape[0] for b in batch], dtype=torch.long)
    y_lengths = torch.tensor([b[1].shape[-1] for b in batch], dtype=torch.long)
    x = torch.zeros(len(batch), int(x_lengths.max()), dtype=torch.long)
    y = torch.zeros(len(batch), N_MELS, int(fix_len_compatibility(int(y_lengths.max()))))
    for i, (xi, yi) in enumerate(batch):
        x[i, :xi.shape[0]] = xi
        y[i, :, :yi.shape[-1]] = yi
    return x, x_lengths, y, y_lengths


class BucketBatchSampler:
    """Gom các câu dài gần nhau vào cùng batch để giảm padding."""

    def __init__(self, lengths, batch_size, bucket_batches=50):
        self.lengths, self.batch_size = lengths, batch_size
        self.chunk = batch_size * bucket_batches

    def __iter__(self):
        idx = list(range(len(self.lengths)))
        random.shuffle(idx)
        batches = []
        for s in range(0, len(idx), self.chunk):
            group = sorted(idx[s:s + self.chunk], key=lambda i: self.lengths[i])
            batches += [group[j:j + self.batch_size] for j in range(0, len(group), self.batch_size)]
        random.shuffle(batches)
        return iter(batches)

    def __len__(self):
        n = len(self.lengths)
        return sum(math.ceil(min(self.chunk, n - s) / self.batch_size) for s in range(0, n, self.chunk))


train_set, val_set = MelDataset(data['train']), MelDataset(data['val'])
train_loader = DataLoader(train_set, batch_sampler=BucketBatchSampler(train_set.lengths, BATCH_SIZE),
                          collate_fn=collate, pin_memory=HAS_GPU)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, collate_fn=collate, pin_memory=HAS_GPU)

start_epoch, global_step = 1, 0
if os.path.exists(RESUME_CKPT):
    ckpt = torch.load(RESUME_CKPT, map_location='cpu', weights_only=False)
    hparams = ckpt['hparams']
    model = build_model(hparams)
    model.load_state_dict(ckpt['state_dict'])
    start_epoch, global_step = ckpt['epoch'] + 1, ckpt['step']
    print(f'[RESUME] epoch {start_epoch}, step {global_step}')
else:
    ckpt = {}
    lj_ckpt = torch.load(download(MATCHA_LJSPEECH_URL, 'matcha_ljspeech.ckpt'), map_location='cpu', weights_only=False)
    hparams = {k: v for k, v in lj_ckpt['hyper_parameters'].items() if k not in ('optimizer', 'scheduler')}
    hparams = OmegaConf.to_container(OmegaConf.create(hparams), resolve=True)
    hparams.update(n_vocab=len(meta['vocab']), n_spks=1, out_size=None,
                   data_statistics={'mel_mean': meta['mel_mean'], 'mel_std': meta['mel_std']})
    model = build_model(hparams)
    # Bộ token khác LJSpeech: chỉ nạp các tensor cùng tên và kích thước
    own_state = model.state_dict()
    reused = {k: v for k, v in lj_ckpt['state_dict'].items()
              if k in own_state and own_state[k].shape == v.shape and k not in ('mel_mean', 'mel_std')}
    model.load_state_dict(reused, strict=False)
    print(f'[INIT] {len(reused)}/{len(own_state)} tensor từ LJSpeech')
    del lj_ckpt

model = model.to(DEVICE).train()
ema = copy.deepcopy(model).eval().requires_grad_(False)
ema_steps = 0
if 'ema' in ckpt:
    ema.load_state_dict(ckpt['ema'])
    ema_steps = ckpt['ema_steps']
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP and AMP_DTYPE == torch.float16)
if 'optimizer' in ckpt:
    optimizer.load_state_dict(ckpt['optimizer'])
    scaler.load_state_dict(ckpt['scaler'])
del ckpt

vocoder, denoiser = load_vocoder()
print(f'{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M tham số | {len(train_loader)} step/epoch')


@torch.no_grad()
def update_ema():
    global ema_steps
    ema_steps += 1
    decay = min(EMA_DECAY, (1 + ema_steps) / (10 + ema_steps))
    for e, p in zip(ema.parameters(), model.parameters()):
        e.lerp_(p, 1 - decay)


def run_batch(batch):
    x, x_lengths, y, y_lengths = (t.to(DEVICE, non_blocking=True) for t in batch)
    with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
        dur_loss, prior_loss, diff_loss, _ = model(x, x_lengths, y, y_lengths)
    return dur_loss.float(), prior_loss.float(), diff_loss.float()


@torch.no_grad()
def evaluate():
    model.eval()
    totals = np.zeros(3)
    for batch in val_loader:
        totals += [l.item() for l in run_batch(batch)]
    model.train()
    return totals / len(val_loader)


def snapshot(epoch):
    return {'hparams': hparams, 'state_dict': ema.state_dict(), 'meta': meta, 'epoch': epoch, 'step': global_step}


epoch, last_resume_save = start_epoch, time.time()
while global_step < TOTAL_STEPS:
    t0 = time.time()
    pbar = tqdm(train_loader, desc=f'Epoch {epoch} (step {global_step}/{TOTAL_STEPS})')
    epoch_losses = []
    for batch in pbar:
        dur_loss, prior_loss, diff_loss = run_batch(batch)
        loss = dur_loss + prior_loss + diff_loss
        optimizer.zero_grad(set_to_none=True)
        if not torch.isfinite(loss):
            continue
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        update_ema()
        global_step += 1
        epoch_losses.append(loss.item())
        pbar.set_postfix(loss=f'{loss.item():.3f}', dur=f'{dur_loss.item():.3f}',
                         prior=f'{prior_loss.item():.3f}', diff=f'{diff_loss.item():.3f}')

        if global_step % SNAPSHOT_EVERY == 0:
            save_atomic(snapshot(epoch), f'{SNAPSHOT_DIR}/step_{global_step:06d}.pt')
        if global_step % SAMPLE_EVERY == 0:
            show_audio(synthesize_vi(ema, EVAL_SENTENCES[0], vocoder, denoiser), f'{SAMPLES_DIR}/step_{global_step:06d}.wav')
        if global_step >= TOTAL_STEPS:
            break

    val_dur, val_prior, val_diff = evaluate()
    train_loss = float(np.mean(epoch_losses)) if epoch_losses else 0.0
    minutes = (time.time() - t0) / 60
    print(f'epoch {epoch:04d} | step {global_step:06d} | {minutes:.1f} phút | train {train_loss:.4f} | '
          f'val {val_dur + val_prior + val_diff:.4f} (dur {val_dur:.3f}, prior {val_prior:.3f}, diff {val_diff:.3f})', flush=True)
    new_file = not os.path.exists(LOG_CSV)
    with open(LOG_CSV, 'a', encoding='utf-8') as f:
        if new_file:
            f.write('timestamp,epoch,step,train_loss,val_dur,val_prior,val_diff,minutes\n')
        f.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')},{epoch},{global_step},{train_loss:.4f},"
                f'{val_dur:.4f},{val_prior:.4f},{val_diff:.4f},{minutes:.2f}\n')

    if time.time() - last_resume_save >= RESUME_EVERY_MIN * 60 or global_step >= TOTAL_STEPS:
        save_atomic({**snapshot(epoch), 'state_dict': model.state_dict(), 'ema': ema.state_dict(),
                     'ema_steps': ema_steps, 'optimizer': optimizer.state_dict(), 'scaler': scaler.state_dict()},
                    RESUME_CKPT)
        last_resume_save = time.time()
    epoch += 1

print(f'[OK] {global_step} step')

In [ ]:
from scipy.signal import resample_poly
from transformers import pipeline

CHOSEN_SNAPSHOT = None   # vd. f'{SNAPSHOT_DIR}/step_150000.pt' nếu nghe thấy snapshot đó tự nhiên hơn
REEVALUATE = False       # True để chấm lại khi đã có matcha_vi.pt

asr = pipeline('automatic-speech-recognition', model='vinai/PhoWhisper-base', device=0 if HAS_GPU else -1,
               generate_kwargs={'language': 'vi', 'task': 'transcribe'})


def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


def cer(ref, hyp):
    ref, hyp = vt.clean_chars(ref), vt.clean_chars(hyp)
    return edit_distance(ref, hyp) / max(1, len(ref))


eval_vocoder, eval_denoiser = load_vocoder()
if CHOSEN_SNAPSHOT:
    shutil.copy(CHOSEN_SNAPSHOT, FINAL_CKPT)
elif REEVALUATE or not os.path.exists(FINAL_CKPT):
    snapshot_files = sorted(glob.glob(f'{SNAPSHOT_DIR}/step_*.pt'))[-N_EVAL_SNAPSHOTS:]
    if not snapshot_files:
        raise FileNotFoundError(f'Chưa có snapshot trong {SNAPSHOT_DIR}: Add Data output của phiên train trước.')
    torch.manual_seed(0)
    results = []
    for path in snapshot_files:
        tts, ck = load_matcha(path)
        errs = [cer(s, asr(resample_poly(synthesize_vi(tts, s, eval_vocoder, eval_denoiser), 16000, SAMPLE_RATE)
                           .astype(np.float32))['text']) for s in EVAL_SENTENCES]
        results.append((float(np.mean(errs)), ck['step'], path))
        print(f"step {ck['step']:>7}: CER {results[-1][0]:.3f}", flush=True)
        del tts
    with open(CER_CSV, 'w', encoding='utf-8') as f:
        f.write('step,cer,path\n')
        f.writelines(f'{s},{c:.4f},{p}\n' for c, s, p in results)
    # CER thấp nhất; bằng nhau thì ưu tiên snapshot train lâu hơn
    best_cer, _, best_path = min(results, key=lambda r: (round(r[0], 3), -r[1]))
    shutil.copy(best_path, FINAL_CKPT)
    print(f'{os.path.basename(best_path)} -> {FINAL_CKPT} (CER {best_cer:.3f})')

tts, _ = load_matcha(FINAL_CKPT)
for i, sentence in enumerate(EVAL_SENTENCES):
    print(sentence)
    show_audio(synthesize_vi(tts, sentence, eval_vocoder, eval_denoiser), f'{SAMPLES_DIR}/final_univ_{i}.wav')

In [ ]:
import torch.nn.functional as F
import matcha.utils.monotonic_align as monotonic_align
from matcha.utils.model import denormalize, sequence_mask

voc_items = torch.load(VOC_PACK, weights_only=False)
tts, tts_ckpt = load_matcha(FINAL_CKPT)


@torch.no_grad()
def teacher_forced_mel(model, text, wav):
    """Mel do decoder sinh ra theo đúng căn chỉnh MAS của câu thật -> khớp từng khung với wav."""
    y = (compute_mel(wav.to(DEVICE)) - model.mel_mean) / model.mel_std
    n_frames = y.shape[-1]
    x = torch.tensor(text_ids(text), dtype=torch.long, device=DEVICE)[None]
    x_lengths = torch.tensor([x.shape[-1]], dtype=torch.long, device=DEVICE)
    mu_x, _, x_mask = model.encoder(x, x_lengths, None)

    padded = fix_len_compatibility(n_frames)
    y = F.pad(y, (0, padded - n_frames))
    y_mask = sequence_mask(torch.tensor([n_frames], device=DEVICE), padded).unsqueeze(1).to(x_mask.dtype)
    attn_mask = x_mask.unsqueeze(-1) * y_mask.unsqueeze(2)
    factor = -0.5 * torch.ones_like(mu_x)
    log_prior = (torch.matmul(factor.transpose(1, 2), y ** 2)
                 - torch.matmul(2.0 * (factor * mu_x).transpose(1, 2), y)
                 + torch.sum(factor * mu_x ** 2, 1).unsqueeze(-1)
                 - 0.5 * math.log(2 * math.pi) * model.n_feats)
    attn = monotonic_align.maximum_path(log_prior, attn_mask.squeeze(1))
    mu_y = torch.matmul(attn.transpose(1, 2), mu_x.transpose(1, 2)).transpose(1, 2)
    mel = model.decoder(mu_y, y_mask, VOC_TF_TIMESTEPS, 0.667, None)
    return denormalize(mel, model.mel_mean, model.mel_std)[0, :, :n_frames]


tf_pack = torch.load(VOC_TF_PACK, weights_only=False) if os.path.exists(VOC_TF_PACK) else None
if tf_pack is None or tf_pack['source_step'] != tts_ckpt['step']:
    pairs = []
    for it in tqdm(voc_items, desc='Mel teacher-forced'):
        mel = teacher_forced_mel(tts, it['text'], it['wav'].float() / 32768.0)
        pairs.append({'mel': mel.half().cpu(), 'wav': it['wav'][:mel.shape[-1] * HOP]})
    tf_pack = {'source_step': tts_ckpt['step'], 'pairs': pairs}
    save_atomic(tf_pack, VOC_TF_PACK)
del tts
print(f"{len(tf_pack['pairs'])} cặp (mel sinh ra, wav thật), step {tf_pack['source_step']}")

In [ ]:
import itertools

from matcha.hifigan.models import (MultiPeriodDiscriminator, MultiScaleDiscriminator, discriminator_loss,
                                   feature_loss, generator_loss)

SEG = HIFIGAN_CONFIG['segment_size']
SEG_FRAMES = SEG // HOP
LOG_SILENCE = math.log(1e-5)


class VocDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        mel, wav = self.pairs[i]['mel'].float(), self.pairs[i]['wav'].float() / 32768.0
        if mel.shape[-1] >= SEG_FRAMES:
            start = random.randint(0, mel.shape[-1] - SEG_FRAMES)
            mel, wav = mel[:, start:start + SEG_FRAMES], wav[start * HOP:(start + SEG_FRAMES) * HOP]
        return F.pad(mel, (0, SEG_FRAMES - mel.shape[-1]), value=LOG_SILENCE), F.pad(wav, (0, SEG - wav.shape[-1]))


voc_loader = DataLoader(VocDataset(tf_pack['pairs']), batch_size=VOC_BATCH, shuffle=True, drop_last=True,
                        num_workers=2 if HAS_GPU else 0, pin_memory=HAS_GPU)

generator = HiFiGAN(AttrDict(HIFIGAN_CONFIG)).to(DEVICE)
mpd, msd = MultiPeriodDiscriminator().to(DEVICE), MultiScaleDiscriminator().to(DEVICE)
optim_g = torch.optim.AdamW(generator.parameters(), VOC_LR, betas=(0.8, 0.99))
optim_d = torch.optim.AdamW(itertools.chain(msd.parameters(), mpd.parameters()), VOC_LR, betas=(0.8, 0.99))
voc_step = 0
if os.path.exists(VOC_RESUME):
    state = torch.load(VOC_RESUME, map_location=DEVICE, weights_only=False)
    if state['source_step'] == tf_pack['source_step']:
        for module, key in ((generator, 'generator'), (mpd, 'mpd'), (msd, 'msd'), (optim_g, 'optim_g'), (optim_d, 'optim_d')):
            module.load_state_dict(state[key])
        voc_step = state['step']
        print(f'[RESUME] vocoder step {voc_step}')
    del state
if voc_step == 0:
    generator.load_state_dict(torch.load(download(HIFIGAN_UNIV_URL, 'hifigan_universal.pt'), map_location=DEVICE)['generator'])
generator.train(), mpd.train(), msd.train()


def save_vocoder():
    save_atomic({'generator': generator.state_dict(), 'mpd': mpd.state_dict(), 'msd': msd.state_dict(),
                 'optim_g': optim_g.state_dict(), 'optim_d': optim_d.state_dict(), 'step': voc_step,
                 'source_step': tf_pack['source_step']}, VOC_RESUME)
    save_atomic({'generator': generator.state_dict(), 'step': voc_step, 'source_step': tf_pack['source_step']}, FINAL_VOC)


tts, _ = load_matcha(FINAL_CKPT)
last_voc_save = time.time()
pbar = tqdm(total=VOC_STEPS, initial=voc_step, desc='HiFi-GAN')
while voc_step < VOC_STEPS:
    for mel, wav in voc_loader:
        mel, y = mel.to(DEVICE, non_blocking=True), wav.to(DEVICE, non_blocking=True).unsqueeze(1)
        y_hat = generator(mel)
        y_mel = compute_mel(y.squeeze(1), fmax=None)
        y_hat_mel = compute_mel(y_hat.squeeze(1), fmax=None)

        optim_d.zero_grad(set_to_none=True)
        y_df_r, y_df_g, _, _ = mpd(y, y_hat.detach())
        y_ds_r, y_ds_g, _, _ = msd(y, y_hat.detach())
        loss_d = discriminator_loss(y_df_r, y_df_g)[0] + discriminator_loss(y_ds_r, y_ds_g)[0]
        loss_d.backward()
        optim_d.step()

        optim_g.zero_grad(set_to_none=True)
        loss_mel = F.l1_loss(y_mel, y_hat_mel) * 45
        loss_g = loss_mel
        if voc_step >= VOC_DISC_WARMUP:
            _, y_df_g, fmap_f_r, fmap_f_g = mpd(y, y_hat)
            _, y_ds_g, fmap_s_r, fmap_s_g = msd(y, y_hat)
            loss_g = (loss_mel + feature_loss(fmap_f_r, fmap_f_g) + feature_loss(fmap_s_r, fmap_s_g)
                      + generator_loss(y_df_g)[0] + generator_loss(y_ds_g)[0])
        loss_g.backward()
        optim_g.step()

        voc_step += 1
        pbar.update(1)
        pbar.set_postfix(mel=f'{loss_mel.item() / 45:.3f}', d=f'{loss_d.item():.3f}', g=f'{loss_g.item():.3f}')
        if voc_step % 500 == 0 or voc_step >= VOC_STEPS:
            new_file = not os.path.exists(VOC_LOG_CSV)
            with open(VOC_LOG_CSV, 'a', encoding='utf-8') as f:
                if new_file:
                    f.write('timestamp,step,mel_loss,disc_loss,gen_loss\n')
                f.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')},{voc_step},{loss_mel.item() / 45:.4f},"
                        f'{loss_d.item():.4f},{loss_g.item():.4f}\n')
        if time.time() - last_voc_save >= VOC_RESUME_EVERY_MIN * 60:
            save_vocoder()
            last_voc_save = time.time()
        if voc_step % VOC_SAMPLE_EVERY == 0:
            generator.eval()
            show_audio(synthesize_vi(tts, EVAL_SENTENCES[0], generator), f'{SAMPLES_DIR}/vocoder_step_{voc_step:06d}.wav')
            generator.train()
        if voc_step >= VOC_STEPS:
            break
save_vocoder()
pbar.close()
print(f'[OK] vocoder {voc_step} step -> {FINAL_VOC}')

In [ ]:
tts, ck = load_matcha(FINAL_CKPT)
voc_univ, den_univ = load_vocoder()
voc_ft, den_ft = load_vocoder(FINAL_VOC)
print(f"Matcha step {ck['step']}")
for i, sentence in enumerate(EVAL_SENTENCES):
    print(f'\n{sentence}\n  HiFi-GAN Universal:')
    show_audio(synthesize_vi(tts, sentence, voc_univ, den_univ), f'{SAMPLES_DIR}/compare_{i}_univ.wav')
    print('  HiFi-GAN fine-tune:')
    show_audio(synthesize_vi(tts, sentence, voc_ft, den_ft), f'{SAMPLES_DIR}/compare_{i}_finetune.wav')
print(f'\nTải về checkpoints/ của dự án:\n  {FINAL_CKPT}\n  {FINAL_VOC}')